In [ ]:
# === 1. Conexión y datos ===
import getpass, os, warnings, itertools
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

warnings.filterwarnings("ignore")
os.makedirs("resultados_06", exist_ok=True)

clave_pg = getpass.getpass("Contraseña de PostgreSQL: ")
engine = create_engine(URL.create(
    "postgresql+psycopg2", username="postgres", password=clave_pg,
    host="localhost", port=5432, database="tesis_agronomia"))

tablas = ["estudiantes", "actas_cursada", "examenes", "insc_cursada",
          "insc_examen", "trayectorias", "indicadores", "grupos", "cursadas"]
datos = {t: pd.read_sql(f"SELECT * FROM tesis.{t}", engine) for t in tablas}
for t, df in datos.items():
    print(f"{t}: {len(df)} filas")

# Nombres anonimizados de las sedes (en ningún resultado aparece el nombre real).
# Los nombres reales están en config_local.py, que no se publica en el repositorio.
from config_local import SEDE_CENTRAL, SEDE_INTERIOR
ETIQ_SEDE = {SEDE_INTERIOR: "Sede del interior", SEDE_CENTRAL: "Sede Central"}
for t in ["estudiantes", "trayectorias", "indicadores"]:
    faltan = set(datos[t]["sede_ingreso"].unique()) - set(ETIQ_SEDE)
    assert not faltan, f"Falta etiqueta de sede en {t}: {faltan}"
    datos[t]["sede_ingreso"] = datos[t]["sede_ingreso"].map(ETIQ_SEDE)

RESUMEN = []                      # líneas para resumen_06.txt
def anotar(*partes):
    linea = " ".join(str(p) for p in partes)
    print(linea)
    RESUMEN.append(linea)

def guardar(df, nombre, indice=True):
    df.to_csv(f"resultados_06/{nombre}.csv", sep=";", decimal=",", index=indice)

In [ ]:
# === 1b. Funciones de 02_preparacion (mismas reglas) ===
CORTE = 21          # 2.º cuatrimestre de 2025
N = 3               # cuatrimestres sin actividad para considerar deserción
W = CORTE - N       # ventana común: último cuatrimestre en que la deserción puede determinarse para todos (1.er cuatr. 2024)

def num_cuatri(anio, c):
    return (anio - 2015) * 2 + (c - 1)

def por_fecha(fecha):
    f = pd.Timestamp(fecha)
    if 4 <= f.month <= 8:
        return num_cuatri(f.year, 1)
    if f.month >= 9:
        return num_cuatri(f.year, 2)
    return num_cuatri(f.year - 1, 2)

def cuatris_de_cursada(periodo, anio, fecha):
    p = str(periodo).lower()
    if "variable" in p:
        return [por_fecha(fecha)]
    if "anual" in p:
        return [num_cuatri(anio, 1), num_cuatri(anio, 2)]
    if "bimestre" in p or p.startswith("1"):
        return [num_cuatri(anio, 1)]
    if p.startswith("2"):
        return [num_cuatri(anio, 2)]
    return [None]

est = datos["estudiantes"].copy()
est["inicio"] = est["cohorte"].apply(lambda a: num_cuatri(a, 1))

def filas_cursada(df):
    filas = []
    for idx, per, anio, fe in zip(df["id_est"], df["periodo"], df["anio_academico"], df["fecha"]):
        for q in cuatris_de_cursada(per, anio, fe):
            filas.append((idx, q))
    return filas

def filas_fecha(df):
    return [(idx, por_fecha(fe)) for idx, fe in zip(df["id_est"], df["fecha"])]

def recortar(filas):
    act = pd.DataFrame(filas, columns=["id_est", "cuatri"]).dropna()
    act = act.astype({"cuatri": int}).merge(est[["id_est", "inicio"]], on="id_est")
    act = act[(act["cuatri"] >= act["inicio"]) & (act["cuatri"] <= CORTE)]
    return act[["id_est", "cuatri"]].drop_duplicates()

# Definición original: inscripciones activas + actas + exámenes + inscripciones a examen
act_original = recortar(filas_cursada(datos["actas_cursada"]) + filas_cursada(datos["insc_cursada"])
                        + filas_fecha(datos["examenes"]) + filas_fecha(datos["insc_examen"]))

# Definición por resultados: solo actas de cursada con resultado (regular, promoción o libre)
# y exámenes rendidos (aprobado o desaprobado). Se excluyen inscripciones y ausentes.
ac_res = datos["actas_cursada"][datos["actas_cursada"]["resultado"].isin(["A", "R"])]
ex_res = datos["examenes"][datos["examenes"]["resultado"].isin(["A", "R"])]
act_resultados = recortar(filas_cursada(ac_res) + filas_fecha(ex_res))

anotar("Estudiante-cuatrimestre con actividad | original:", len(act_original),
       "(esperado 9943) | solo resultados:", len(act_resultados))

In [ ]:
# === 1c. Construcción de cada escenario ===
egreso_cuatri = est.set_index("id_est")["fecha_egreso"].apply(
    lambda f: por_fecha(f) if pd.notna(f) else np.nan)

def escenario(actividad, ventana=False):
    """Devuelve una fila por estudiante con grupo, tiempo y evento, con las reglas de 02_preparacion."""
    b = est.set_index("id_est")[["cohorte", "sede_ingreso", "inicio"]].copy()
    b["ultima"] = actividad.groupby("id_est")["cuatri"].max()
    b["sin_actividad"] = b["ultima"].isna()
    egresado = egreso_cuatri.reindex(b.index).notna()
    b["deserta"] = (~egresado) & (CORTE - b["ultima"] >= N)
    b["deserta_1er_anio"] = b["deserta"] & (b["ultima"] <= b["inicio"] + 1)
    b["grupo"] = np.where(b["sin_actividad"], "sin actividad",
                  np.where(b["deserta_1er_anio"], "desertó en 1.er año", "modelo"))
    fin = egreso_cuatri.reindex(b.index).fillna(b["ultima"])
    if ventana:
        fin = np.minimum(fin, W)                    # todos los no desertores se censuran en W
    b["tiempo"] = fin - (b["inicio"] + 1)
    b["evento"] = b["deserta"].astype(float)
    b.loc[b["grupo"] != "modelo", ["tiempo", "evento"]] = np.nan
    return b

ESC = {
    "original":            escenario(act_original),
    "solo_resultados":     escenario(act_resultados),
    "ventana_comun":       escenario(act_original, ventana=True),
    "resultados_y_ventana": escenario(act_resultados, ventana=True),
}

# Control: el escenario original debe reproducir tesis.trayectorias
tr0 = datos["trayectorias"].set_index("id_est")
o = ESC["original"]
tr0 = tr0.reindex(o.index)
iguales = ((o["grupo"] == tr0["grupo"]).all()
           and np.allclose(o["tiempo"].fillna(-1), tr0["tiempo"].fillna(-1))
           and np.allclose(o["evento"].fillna(-1), tr0["evento"].astype(float).fillna(-1)))
anotar("Control: el escenario original reproduce tesis.trayectorias:", iguales)
if not iguales:
    print("ATENCIÓN: revisar antes de seguir. Diferencias de grupo:",
          (o["grupo"] != tr0["grupo"]).sum())

In [ ]:
# === 2. Deserción en el primer año según la definición de actividad ===
filas = []
for nombre in ["original", "solo_resultados"]:
    b = ESC[nombre]
    for coh, g in list(b.groupby("cohorte")) + [("Total", b)]:
        n_tot = len(g)
        sin = int(g["sin_actividad"].sum())
        d1 = int((g["grupo"] == "desertó en 1.er año").sum())
        filas.append({"definicion": nombre, "cohorte": coh, "ingresantes": n_tot,
                      "sin_resultados": sin, "desertores_1er_anio": d1,
                      "pct_1er_anio_incl_sin_resultados": 100 * (d1 + sin) / n_tot,
                      "pct_1er_anio_entre_quienes_empezaron": 100 * d1 / (n_tot - sin),
                      "grupo_modelo": int((g["grupo"] == "modelo").sum())})
t2 = pd.DataFrame(filas).round(1)
guardar(t2, "2_desercion_primer_anio", indice=False)
print(t2.to_string(index=False))
for nombre in ["original", "solo_resultados"]:
    r = t2[(t2["definicion"] == nombre) & (t2["cohorte"] == "Total")].iloc[0]
    anotar(f"[2] {nombre}: ingresantes {r.ingresantes}, sin ningún resultado {r.sin_resultados}, "
           f"deserción 1.er año incl. sin resultados {r.pct_1er_anio_incl_sin_resultados} %, "
           f"entre quienes empezaron {r.pct_1er_anio_entre_quienes_empezaron} %, grupo del modelo {r.grupo_modelo}")

# Cómo se reclasifica cada estudiante al pasar a «solo resultados»
cruce = pd.crosstab(ESC["original"]["grupo"], ESC["solo_resultados"]["grupo"],
                    rownames=["original"], colnames=["solo resultados"], margins=True)
guardar(cruce, "2_cruce_grupos")
print("\n", cruce)
anotar("[2] Cruce de grupos (filas: original; columnas: solo resultados):\n" + cruce.to_string())

# Perfil Participación baja: ¿qué pasa con sus 324 estudiantes?
gr = datos["grupos"].set_index("id_est")
pb = gr.index[gr["perfil"] == "Participación baja"]
reclas = ESC["solo_resultados"].loc[pb, "grupo"].value_counts()
anotar(f"[2] Participación baja ({len(pb)}) con actividad solo por resultados:", reclas.to_dict())

In [ ]:
# === 4a. Kaplan-Meier por escenario ===
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.compare import compare_survival
from sksurv.util import Surv

def km_en(df, puntos=(2, 4, 8)):
    df = df.dropna(subset=["tiempo"])
    if len(df) == 0:
        return {f"S({p})": np.nan for p in puntos}
    t, s = kaplan_meier_estimator(df["evento"].astype(bool), df["tiempo"])
    t, s = np.r_[0, t], np.r_[1, s]
    res = {}
    for p in puntos:
        res[f"S({p})"] = s[t <= p][-1] if df["tiempo"].max() >= p else np.nan
    return res

filas = []
for nombre, b in ESC.items():
    m = b[b["grupo"] == "modelo"]
    filas.append({"escenario": nombre, "corte": "Total", "n": len(m),
                  "desertores": int(m["evento"].sum()), **km_en(m)})
    for sede, g in m.groupby("sede_ingreso"):
        filas.append({"escenario": nombre, "corte": sede, "n": len(g),
                      "desertores": int(g["evento"].sum()), **km_en(g)})
    for coh, g in m.groupby("cohorte"):
        filas.append({"escenario": nombre, "corte": f"Cohorte {coh}", "n": len(g),
                      "desertores": int(g["evento"].sum()), **km_en(g)})
t4a = pd.DataFrame(filas)
guardar(t4a.round(3), "4a_kaplan_meier", indice=False)
print(t4a.round(3).to_string(index=False))
for nombre in ESC:
    r = t4a[(t4a["escenario"] == nombre) & (t4a["corte"] == "Total")].iloc[0]
    sedes = t4a[(t4a["escenario"] == nombre) & t4a["corte"].str.startswith("Sede")]
    c23 = t4a[(t4a["escenario"] == nombre) & (t4a["corte"] == "Cohorte 2023")].iloc[0]
    anotar(f"[4a] {nombre}: n={r.n}, desertores={r.desertores}, S(4)={r['S(4)']:.3f}, S(8)={r['S(8)']:.3f}; "
           + "; ".join(f"{x['corte']} S(4)={x['S(4)']:.3f}" for _, x in sedes.iterrows())
           + f"; cohorte 2023 S(2)={c23['S(2)']:.3f} S(4)={c23['S(4)']:.3f}")

# Perfiles con «solo resultados» (los que siguen en el grupo del modelo)
m = ESC["solo_resultados"].join(gr[["perfil"]], how="inner")
m = m[m["grupo"] == "modelo"]
for perfil, g in m.groupby("perfil"):
    anotar(f"[4a] solo_resultados, perfil {perfil}: n={len(g)}, S(4)={km_en(g)['S(4)']:.3f}")

In [ ]:
# === 4b. Cox con todos los casos (inferencia), por escenario ===
from statsmodels.duration.hazard_regression import PHReg
from scipy import stats

ind = datos["indicadores"]
ind = ind[ind["incluido"]].set_index("id_est")

def matriz(b):
    """Variables del modelo de 05_supervivencia para los estudiantes con indicadores del grupo del modelo."""
    m = b[b["grupo"] == "modelo"].join(
        ind[["materias_cursadas", "prop_ausentes", "materias_aprobadas", "ingreso_equiv"]], how="inner")
    m = m[m["tiempo"] > 0]
    periodo = pd.cut(m["cohorte"], bins=[2014, 2018, 2021, 2023],
                     labels=["2015-2018", "2019-2021", "2022-2023"])
    X = pd.DataFrame({
        "materias_cursadas":  m["materias_cursadas"],
        "ausentes_x10":       m["prop_ausentes"] * 10,
        "materias_aprobadas": m["materias_aprobadas"],
        "sede_sp":            (m["sede_ingreso"] == "Sede del interior").astype(int),
        "coh_2019_2021":      (periodo == "2019-2021").astype(int),
        "coh_2022_2023":      (periodo == "2022-2023").astype(int),
        "ingreso_equiv":      m["ingreso_equiv"].astype(int),
    }, index=m.index)
    y = Surv.from_arrays(event=m["evento"].astype(bool), time=m["tiempo"].astype(float))
    return X, y, m

def prueba_ph(res, y, columnas):
    ev = y["event"]
    r = np.asarray(res.schoenfeld_residuals)[ev]
    t = y["time"][ev]
    d = ev.sum()
    V = np.asarray(res.cov_params())
    r_esc = d * r @ V
    gc = stats.rankdata(t); gc = gc - gc.mean()
    chi2 = (gc @ r_esc) ** 2 / (d * np.diag(V) * (gc ** 2).sum())
    return stats.chi2.sf(chi2, df=1)

def cox_tabla(X, y):
    res = PHReg(endog=y["time"], exog=X, status=y["event"].astype(int), ties="efron").fit()
    ic = np.exp(np.asarray(res.conf_int()))
    return pd.DataFrame({"HR": np.exp(np.asarray(res.params)), "IC95_inf": ic[:, 0],
                         "IC95_sup": ic[:, 1], "p": np.asarray(res.pvalues),
                         "p_RP": prueba_ph(res, y, X.columns)}, index=X.columns)

MAT = {nombre: matriz(b) for nombre, b in ESC.items()}
COX = {}
for nombre, (X, y, m) in MAT.items():
    COX[nombre] = cox_tabla(X, y)
    guardar(COX[nombre].round(3), f"4b_cox_completo_{nombre}")
    print(f"\n--- Cox con todos los casos: {nombre} (n = {len(X)}, eventos = {int(y['event'].sum())}) ---")
    print(COX[nombre].round(3))
    anotar(f"[4b] Cox completo {nombre}: n={len(X)}, eventos={int(y['event'].sum())}; "
           + "; ".join(f"{v} HR={r.HR:.2f} ({r.IC95_inf:.2f}-{r.IC95_sup:.2f}) p={r.p:.3f} pRP={r.p_RP:.3f}"
                       for v, r in COX[nombre].iterrows()))

In [ ]:
# === 4c. Cox extendido (tramos) con todos los casos, por escenario ===
cortes = [(0, 3, "t1_3"), (3, 6, "t4_6"), (6, 20, "t7_20")]
vars_ext = ["materias_cursadas", "ausentes_x10", "materias_aprobadas", "coh_2022_2023", "ingreso_equiv",
            "sede_t1_3", "sede_t4_6", "sede_t7_20", "coh1921_t1_3", "coh1921_t4_6", "coh1921_t7_20"]

def episodios(X, y):
    filas = []
    Xr = X.reset_index(drop=True)
    for i in range(len(Xr)):
        T, E = y["time"][i], y["event"][i]
        for a, b_, nom in cortes:
            if T > a:
                f = Xr.iloc[i].to_dict()
                f.update(entrada=a, salida=min(T, b_), evento=int(E and T <= b_), tramo=nom)
                filas.append(f)
    ep = pd.DataFrame(filas)
    for nom in ["t1_3", "t4_6", "t7_20"]:
        en = (ep["tramo"] == nom).astype(int)
        ep[f"sede_{nom}"] = ep["sede_sp"] * en
        ep[f"coh1921_{nom}"] = ep["coh_2019_2021"] * en
    return ep

EXT = {}
for nombre, (X, y, m) in MAT.items():
    ep = episodios(X, y)
    cols = [v for v in vars_ext if ep[v].sum() > 0]          # con ventana común puede no haber casos en t7_20
    res = PHReg(endog=ep["salida"], exog=ep[cols], status=ep["evento"],
                entry=ep["entrada"], ties="efron").fit()
    ic = np.exp(np.asarray(res.conf_int()))
    EXT[nombre] = pd.DataFrame({"HR": np.exp(np.asarray(res.params)), "IC95_inf": ic[:, 0],
                                "IC95_sup": ic[:, 1], "p": np.asarray(res.pvalues)}, index=cols)
    guardar(EXT[nombre].round(3), f"4c_cox_extendido_{nombre}")
    sel = EXT[nombre].loc[[c for c in cols if c.startswith(("sede_", "coh1921_"))]]
    print(f"\n--- Cox extendido con todos los casos: {nombre} ---")
    print(sel.round(3))
    anotar(f"[4c] Cox extendido {nombre}: "
           + "; ".join(f"{v} HR={r.HR:.2f} ({r.IC95_inf:.2f}-{r.IC95_sup:.2f}) p={r.p:.3f}" for v, r in sel.iterrows()))

In [ ]:
# === 5. Modelo de tiempo discreto (log-log complementario) ===
import statsmodels.api as sm

def tiempo_discreto(X, y):
    filas = []
    Xr = X.reset_index(drop=True)
    for i in range(len(Xr)):
        T, E = int(y["time"][i]), bool(y["event"][i])
        for t in range(1, T + 1):
            f = Xr.iloc[i].to_dict()
            f["t"] = t
            f["y"] = int(E and t == T)
            filas.append(f)
    pp = pd.DataFrame(filas)
    pp["t_cat"] = pd.cut(pp["t"], bins=[0, 1, 2, 3, 4, 5, 6, 8, 12, 40],
                         labels=["t1", "t2", "t3", "t4", "t5", "t6", "t7-8", "t9-12", "t13+"])
    dummies = pd.get_dummies(pp["t_cat"], drop_first=False, dtype=float)
    dummies = dummies.loc[:, dummies.sum() > 0]
    Z = pd.concat([dummies, pp[X.columns].astype(float)], axis=1)
    mod = sm.GLM(pp["y"], Z, family=sm.families.Binomial(link=sm.families.links.CLogLog())).fit()
    ic = np.exp(mod.conf_int())
    out = pd.DataFrame({"HR": np.exp(mod.params), "IC95_inf": ic[0], "IC95_sup": ic[1],
                        "p": mod.pvalues}).loc[X.columns]
    return out, len(pp)

DISC = {}
for nombre in ["original", "ventana_comun"]:
    X, y, m = MAT[nombre]
    DISC[nombre], n_pp = tiempo_discreto(X, y)
    guardar(DISC[nombre].round(3), f"5_tiempo_discreto_{nombre}")
    comp = DISC[nombre][["HR", "IC95_inf", "IC95_sup", "p"]].join(COX[nombre][["HR"]], rsuffix="_Cox")
    print(f"\n--- Tiempo discreto: {nombre} ({n_pp} estudiante-cuatrimestres) ---")
    print(comp.round(3))
    anotar(f"[5] Tiempo discreto {nombre}: "
           + "; ".join(f"{v} HR={r.HR:.2f} ({r.IC95_inf:.2f}-{r.IC95_sup:.2f}) p={r.p:.3f} [Cox {r.HR_Cox:.2f}]"
                       for v, r in comp.iterrows()))

In [ ]:
# === 6. Cox frente a RSF con validación cruzada repetida ===
from sklearn.model_selection import RepeatedStratifiedKFold
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw, integrated_brier_score

X, y, m = MAT["original"]
estrato = (pd.Series(y["event"].astype(int), index=X.index).astype(str) + "_" + m["sede_ingreso"]).to_numpy()

def metricas(y_ent, y_pru, riesgo, surv_fns):
    c_h = concordance_index_censored(y_pru["event"], y_pru["time"], riesgo)[0]
    try:
        c_u = concordance_index_ipcw(y_ent, y_pru, riesgo, tau=min(19, y_ent["time"].max() - 1))[0]
    except Exception:
        c_u = np.nan
    try:
        hi = min(19, y_pru["time"].max() - 1, y_ent["time"].max() - 1)
        tiempos = np.arange(max(1, y_pru["time"].min()), hi + 1)
        S = np.vstack([f(tiempos) for f in surv_fns])
        ibs = integrated_brier_score(y_ent, y_pru, S, tiempos)
    except Exception:
        ibs = np.nan
    return c_h, c_u, ibs

rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=2026)
filas = []
for k, (i_ent, i_pru) in enumerate(rkf.split(X, estrato)):
    X_e, X_p, y_e, y_p = X.iloc[i_ent], X.iloc[i_pru], y[i_ent], y[i_pru]
    cox = CoxPHSurvivalAnalysis(ties="efron").fit(X_e, y_e)
    rsf = RandomSurvivalForest(n_estimators=500, min_samples_leaf=30, max_features="sqrt",
                               random_state=2026, n_jobs=-1).fit(X_e, y_e)
    for nombre_m, mod in [("Cox", cox), ("RSF", rsf)]:
        c_h, c_u, ibs = metricas(y_e, y_p, mod.predict(X_p), mod.predict_survival_function(X_p))
        filas.append({"particion": k, "modelo": nombre_m, "C_Harrell": c_h, "C_Uno": c_u, "IBS": ibs})
cv = pd.DataFrame(filas)
guardar(cv, "6_cv_cox_rsf", indice=False)

ancho = cv.pivot(index="particion", columns="modelo")
resumen_cv = cv.groupby("modelo")[["C_Harrell", "C_Uno", "IBS"]].agg(["mean", "std"])
print(resumen_cv.round(3))
anotar("[6] Validación cruzada repetida 5x10, media (desvío): " + "; ".join(
    f"{mo}: C Harrell {resumen_cv.loc[mo, ('C_Harrell', 'mean')]:.3f} ({resumen_cv.loc[mo, ('C_Harrell', 'std')]:.3f}), "
    f"C Uno {resumen_cv.loc[mo, ('C_Uno', 'mean')]:.3f}, IBS {resumen_cv.loc[mo, ('IBS', 'mean')]:.3f}"
    for mo in ["Cox", "RSF"]))
for met in ["C_Harrell", "C_Uno", "IBS"]:
    d = (ancho[(met, "Cox")] - ancho[(met, "RSF")]).dropna()
    mejor = (d > 0).mean() if met != "IBS" else (d < 0).mean()
    lo, hi = np.percentile(d, [2.5, 97.5])
    anotar(f"[6] {met} Cox − RSF: media {d.mean():+.3f}, percentiles 2,5-97,5 [{lo:+.3f}, {hi:+.3f}], "
           f"Cox mejor en el {100 * mejor:.0f} % de {len(d)} particiones")

# Sensibilidad a la configuración del RSF (seis configuraciones de 05, 5 particiones x 2 repeticiones)
rkf2 = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=7)
filas = []
for hoja, mf in itertools.product([5, 15, 30], ["sqrt", None]):
    for i_ent, i_pru in rkf2.split(X, estrato):
        rsf = RandomSurvivalForest(n_estimators=500, min_samples_leaf=hoja, max_features=mf,
                                   random_state=2026, n_jobs=-1).fit(X.iloc[i_ent], y[i_ent])
        c = concordance_index_censored(y[i_pru]["event"], y[i_pru]["time"], rsf.predict(X.iloc[i_pru]))[0]
        filas.append({"min_samples_leaf": hoja, "max_features": "todas" if mf is None else mf, "C_Harrell": c})
conf = pd.DataFrame(filas).groupby(["min_samples_leaf", "max_features"])["C_Harrell"].agg(["mean", "std"]).round(3)
guardar(conf, "6_rsf_configuraciones")
print(conf)
anotar("[6] RSF por configuración (C Harrell media en 10 particiones): "
       + "; ".join(f"hoja {i[0]}, {i[1]}: {r['mean']:.3f}" for i, r in conf.iterrows()))

In [ ]:
# === 7. Peso de la cohorte: concordancia de Cox con y sin cohorte ===
filas = []
for nombre in ["original", "ventana_comun"]:
    X, y, m = MAT[nombre]
    est7 = (pd.Series(y["event"].astype(int), index=X.index).astype(str) + "_" + m["sede_ingreso"]).to_numpy()
    for variante, cols in [("con cohorte", list(X.columns)),
                           ("sin cohorte", [c for c in X.columns if not c.startswith("coh_")])]:
        for k, (i_e, i_p) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=10,
                                                               random_state=2026).split(X, est7)):
            cox = CoxPHSurvivalAnalysis(ties="efron").fit(X.iloc[i_e][cols], y[i_e])
            c = concordance_index_censored(y[i_p]["event"], y[i_p]["time"], cox.predict(X.iloc[i_p][cols]))[0]
            filas.append({"escenario": nombre, "variante": variante, "particion": k, "C_Harrell": c})
t7 = pd.DataFrame(filas)
r7 = t7.groupby(["escenario", "variante"])["C_Harrell"].agg(["mean", "std"]).round(3)
guardar(r7, "7_cohorte_concordancia")
print(r7)
anotar("[7] C de Harrell de Cox (validación cruzada 5x10): "
       + "; ".join(f"{i[0]} {i[1]}: {r['mean']:.3f}" for i, r in r7.iterrows()))

In [ ]:
# === 8. Agrupamiento: estabilidad, k = 2 frente a k = 3, transformación y oferta por sede ===
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score

vars_grupo = ["materias_cursadas", "prop_ausentes", "materias_aprobadas"]
Xg = ind[vars_grupo]
X_esc = StandardScaler().fit_transform(Xg)
km3 = KMeans(n_clusters=3, n_init=20, random_state=42).fit(X_esc)
ref = gr.loc[ind.index, "grupo_km"].to_numpy()
anotar("[8] Control: K-means k=3 reproduce los perfiles de tesis.grupos (ARI):",
       round(adjusted_rand_score(ref, km3.labels_), 3))

# Estabilidad por submuestreo (80 % sin reposición, 200 repeticiones)
rng = np.random.default_rng(2026)
filas = []
base_lab = {}
for k in [2, 3, 4]:
    base_lab[("K-means", k)] = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X_esc)
    base_lab[("Ward", k)] = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X_esc)
for b in range(200):
    idx = rng.choice(len(X_esc), size=int(0.8 * len(X_esc)), replace=False)
    for k in [2, 3, 4]:
        lab_km = KMeans(n_clusters=k, n_init=10, random_state=b).fit_predict(X_esc[idx])
        lab_w = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X_esc[idx])
        filas.append({"k": k, "algoritmo": "K-means", "ARI": adjusted_rand_score(base_lab[("K-means", k)][idx], lab_km)})
        filas.append({"k": k, "algoritmo": "Ward", "ARI": adjusted_rand_score(base_lab[("Ward", k)][idx], lab_w)})
estab = pd.DataFrame(filas).groupby(["algoritmo", "k"])["ARI"].describe(percentiles=[0.05, 0.5]).round(3)
guardar(estab, "8_estabilidad")
print(estab[["mean", "5%", "50%", "min"]])
anotar("[8] Estabilidad por submuestreo (ARI medio y percentil 5): "
       + "; ".join(f"{i[0]} k={i[1]}: {r['mean']:.3f} (p5 {r['5%']:.3f})" for i, r in estab.iterrows()))

# K-means frente a Ward para k = 2, 3 y 4
for k in [2, 3, 4]:
    anotar(f"[8] ARI K-means vs Ward, k={k}:",
           round(adjusted_rand_score(base_lab[("K-means", k)], base_lab[("Ward", k)]), 3))

# Materias aprobadas transformadas: log(1 + x)
Xt = Xg.copy()
Xt["materias_aprobadas"] = np.log1p(Xt["materias_aprobadas"])
Xt_esc = StandardScaler().fit_transform(Xt)
km3_t = KMeans(n_clusters=3, n_init=20, random_state=42).fit(Xt_esc)
anotar("[8] Con log(1 + materias aprobadas): silueta k=3", round(silhouette_score(Xt_esc, km3_t.labels_), 3),
       "| ARI con los perfiles originales", round(adjusted_rand_score(ref, km3_t.labels_), 3))
# k = 2: ¿qué perfiles se unen?
lab2 = base_lab[("K-means", 2)]
cruce_k2 = pd.crosstab(gr.loc[ind.index, "perfil"].to_numpy(), lab2, rownames=["perfil k=3"], colnames=["grupo k=2"])
guardar(cruce_k2, "8_cruce_k2_k3")
anotar("[8] Perfiles k=3 frente a grupos k=2:\n" + cruce_k2.to_string())

# Oferta del 2.º cuatrimestre del primer año por sede
cu = datos["cursadas"].merge(ind[["sede_ingreso"]].reset_index(), on="id_est")
cu = cu.merge(est[["id_est", "inicio"]], on="id_est")
c2 = cu[cu["cuatri"] == cu["inicio"] + 1]
oferta = (c2.groupby(["sede_ingreso", "id_est"])["materia"].nunique()
            .groupby("sede_ingreso").agg(["mean", "median"]).round(2))
distintas = c2.groupby(["sede_ingreso", "anio_academico"])["materia"].nunique().groupby("sede_ingreso").agg(["mean", "min", "max"])
guardar(oferta, "8_cursadas_2c_por_sede")
guardar(distintas, "8_materias_distintas_2c_por_sede")
print(oferta, "\n", distintas)
anotar("[8] Cursadas en el 2.º cuatrimestre del 1.er año por estudiante (media, mediana): "
       + "; ".join(f"{s}: {r['mean']} / {r['median']}" for s, r in oferta.iterrows()))
anotar("[8] Materias distintas cursadas en el 2.º cuatrimestre por año (media, mín., máx.): "
       + "; ".join(f"{s}: {r['mean']:.1f} / {r['min']} / {r['max']}" for s, r in distintas.iterrows()))

In [ ]:
# === 9. Resumen ===
with open("resultados_06/resumen_06.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(RESUMEN))
print("\nListo. Resumen guardado en resultados_06/resumen_06.txt")
print(sorted(os.listdir("resultados_06")))